# فصل ۱۳: پردازش داده‌ها در LangChain (Data Processing)
## بخش ۵: تقسیم‌بندی معنایی متن (Semantic Chunking)

تا این لحظه، تمام اسپلیترهایی که بررسی کردیم (Character و Recursive) همانند **«یک ربات خط‌کش‌به‌دست»** عمل می‌کردند!
آن‌ها صرفاً به طول رشته یا علائم نگارشی نگاه می‌کردند: *«به کاراکتر ۱۰۰۰ رسیدم؟ برش بزن!»* یا *«به دو تا اینتر رسیدم؟ برش بزن!»*

این روش‌ها **ساختار ظاهری (Syntax)** را می‌فهمند، اما از **معنی (Semantics)** کاملاً بی‌خبرند.
فرض کنید در پاراگراف اول درباره آشپزی و در پاراگراف دوم ناگهان درباره فلسفه یونان باستان صحبت شده باشد؛ اسپلیتر بازگشتی ممکن است هر دو موضوع را در یک چانک مخلوط کند (مخلوط قورمه‌سبزی و ارسطو!).

> 🧠 **تقسیم‌بندی معنایی (Semantic Chunking):** در این روش خط‌کش کنار گذاشته می‌شود و متن **نه بر اساس تعداد کاراکتر، بلکه بر اساس تغییر موضوع (Topic Transition)** خرد می‌شود.

### ۱. الگوریتم تقسیم‌بندی معنایی در لایه زیرین چگونه کار می‌کند؟

این روش مستقیماً بر پایه مدل‌های **تعبیه متن (Embedding)** کار می‌کند:

```
متن ورودی
  │
  ▼
[تفکیک به تک‌تک جملات سازنده]
  │
  ▼
[تولید بردار Embedding برای هر جمله]
  │
  ▼
[محاسبه فاصله معنایی (Distance) بین جمله i و جمله i+1]
  │
  ├── فاصله کم (شباهت بالا، مثلا ۹۰٪)  ──>  موضوع عوض نشده؛ به چانک جاری اضافه کن.
  └── فاصله زیاد (جهش ناگهانی)          ──>  🛑 نقطه شکست (Break Point)! موضوع عوض شد؛ برش بزن!
```

### ۲. پیاده‌سازی عملی در LangChain با `SemanticChunker`

برای اجرای این روش، از مدل چندزبانه و محلی **HuggingFace** (`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`) استفاده می‌کنیم تا بدون نیاز به پرداخت هزینه API یا محدودیت اینترنت، جملات را بردارسازی کنیم.

متنی شامل ۳ موضوع کاملاً متفاوت (**نجوم و فضا**، **غذای پیتزا**، و **زبان برنامه‌نویسی پایتون**) را آزمایش می‌کنیم:

In [3]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

# بارگذاری مدل تعبیه متن محلی و چندزبانه
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# متنی مرکب از ۳ موضوع کاملاً مستقل
multi_topic_text = """
منظومه شمسی از خورشید و اجرامی که به دور آن می‌چرخند تشکیل شده است. هشت سیاره بزرگ در مدارهای مشخص قرار دارند. گرانش خورشید همه اجرام را در مدار نگه می‌دارد.
پیتزا یک غذای لذیذ با اصالت ایتالیایی است که از خمیر گندم همراه با سس گوجه فرنگی، پنیر موزارلا و ادویه‌ها تهیه می‌شود. این غذا در فرهای بسیار داغ پخته می‌شود.
پایتون یک زبان برنامه‌نویسی سطح بالا و همه‌منظوره است. فلسفه طراحی پایتون بر خوانایی کد و استفاده از فاصله‌گذاری مناسب تاکید فراوان دارد.
"""

# راه‌اندازی اسپلیتر معنایی با روش آستانه صدک
semantic_splitter = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=85
)

docs = semantic_splitter.create_documents([multi_topic_text.strip()])

print(f"✅ تعداد چانک‌های تولیدشده بر پایه تغییر موضوع: {len(docs)}\n")
for i, doc in enumerate(docs, 1):
    print(f"--- چانک {i} ---")
    print(doc.page_content.strip())
    print("-" * 50)

Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1890.92it/s]


✅ تعداد چانک‌های تولیدشده بر پایه تغییر موضوع: 2

--- چانک 1 ---
منظومه شمسی از خورشید و اجرامی که به دور آن می‌چرخند تشکیل شده است. هشت سیاره بزرگ در مدارهای مشخص قرار دارند. گرانش خورشید همه اجرام را در مدار نگه می‌دارد. پیتزا یک غذای لذیذ با اصالت ایتالیایی است که از خمیر گندم همراه با سس گوجه فرنگی، پنیر موزارلا و ادویه‌ها تهیه می‌شود. این غذا در فرهای بسیار داغ پخته می‌شود.
--------------------------------------------------
--- چانک 2 ---
پایتون یک زبان برنامه‌نویسی سطح بالا و همه‌منظوره است. فلسفه طراحی پایتون بر خوانایی کد و استفاده از فاصله‌گذاری مناسب تاکید فراوان دارد.
--------------------------------------------------


> ✨ **خروجی شگفت‌انگیز را مشاهده کنید:**
> با وجود اینکه هیچ محدودیتی برای تعداد کاراکترها مشخص نکردیم، الگوریتم به درستی تشخیص داد که هر پاراگراف حوزه معنایی متفاوتی دارد و متن را دقیقاً در مرزهای تغییر موضوع برش زد!

### ۳. بررسی ۳ استراتژی تعیین نقطه شکست (`breakpoint_threshold_type`)

فرض کنید الگوریتم میزان تفاوت معنایی بین جملات متوالی را به صورت این لیست عددی محاسبه کرده است:
$$[10, 12, 11, 9, \mathbf{85}, 12, 10]$$
عدد **۸۵** نشان‌دهنده یک پرش موضوعی عظیم است. حال ۳ روش تنظیم آستانه را بررسی می‌کنیم:

| استراتژی | نحوه عملکرد | منطق تصمیم‌گیری | کاربرد مناسب |
| :--- | :--- | :--- | :--- |
| **۱. روش صدک (`percentile`)** | فاصله‌ها مرتب شده و فقط $X$ درصد بزرگترین تغییرات برش می‌خورند. | با مقدار ۹۰، ۹۰ درصد تغییرات ریز نادیده گرفته شده و فقط در ۱۰ درصد جهش‌های بزرگ برش می‌زند. | اسناد عمومی با بخش‌بندی‌های مشخص. |
| **۲. انحراف معیار (`standard_deviation`)** | میانگین تغییرات ($\mu$) و انحراف معیار ($\sigma$) محاسبه می‌شود؛ هر فاصله‌ای فراتر از $\mu + k\sigma$ باشد برش می‌خورد. | با $k=3$ (سه سیگما) فقط جهش‌های بسیار نادر برش می‌خورند. با $k=1$ سیستم حساس‌تر شده و برش‌های بیشتری می‌زند. | متون تخصصی، علمی و مقالات با توزیع آماری پیوسته. |
| **۳. دامنه میان‌چارکی (`interquartile`)** | داده‌ها به ۴ چارک تقسیم شده و نقاط پرت با فرمول $Q_3 + 1.5 \times \text{IQR}$ شناسایی می‌شوند. | همانند شناسایی داده‌های غیرنرمال (Outliers)؛ برش فقط در محل ناهنجاری‌های موضوعی بزرگ. | اسناد حقوقی و کتاب‌های قطور بدون فرمت استاندارد. |

In [4]:
# مقایسه خروجی ۳ نوع آستانه بر روی همان متن
threshold_types = ["percentile", "standard_deviation", "interquartile"]

for t_type in threshold_types:
    splitter_variant = SemanticChunker(
        embeddings=embeddings,
        breakpoint_threshold_type=t_type
    )
    variant_docs = splitter_variant.create_documents([multi_topic_text.strip()])
    print(f"روش '{t_type}': تعداد چانک‌های تولیدشده = {len(variant_docs)}")

روش 'percentile': تعداد چانک‌های تولیدشده = 2
روش 'standard_deviation': تعداد چانک‌های تولیدشده = 1
روش 'interquartile': تعداد چانک‌های تولیدشده = 1


### ۴. توازن مهندسی: چرا همیشه از Semantic Chunking استفاده نمی‌کنیم؟

اگرچه تقسیم‌بندی معنایی بسیار هوشمند است، اما دو محدودیت عمده دارد که مانع از استفاده همیشگی آن می‌شود:

1. **هزینه پردازشی و سرعت (Latency & Cost):**
   - در روش Recursive خرد کردن متن با منطق کاراکتری در کسری از میلی‌ثانیه و بدون هزینه انجام می‌شود.
   - در Semantic Chunking باید **تمام جملات تک‌به‌تک بردارسازی (Embed) شوند** که در اسناد بزرگ و مدل‌های ابری، زمان‌بر و هزینه‌بر است.

2. **طول غیرقابل پیش‌بینی چانک‌ها:**
   - در روش‌های کاراکتری دقیقاً می‌دانید چانک‌ها حدود ۱۰۰۰ کاراکتر هستند.
   - در روش معنایی، ممکن است یک چانک ۳۰ کاراکتر (یک جمله کوتاه منفرد) و چانک دیگر ۴۰۰۰ کاراکتر (یک بحث طولانی که تغییر موضوع نداشته) باشد که دومی ممکن است از پنجره زمینه فراتر رود!

### 📊 مقایسه جامع استراتژی‌های خرد کردن متن در LangChain:

| روش اسپلیتر | منطق کار | سرعت و هزینه | هوشمندی معنایی | کاربرد بهینه |
| :--- | :--- | :--- | :--- | :--- |
| **`CharacterTextSplitter`** | فواصل ثابت کاراکتری | ⚡ فوق‌العاده سریع و رایگان | ❌ صفر (خطر نصف کردن کلمات) | پروژه‌های تستی و خام |
| **`RecursiveCharacterTextSplitter`** | اولویت پاراگراف $\rightarrow$ خط $\rightarrow$ کلمه | ⚡ بسیار سریع و رایگان | 🟡 متوسط (حفظ یکپارچگی کلمات و جملات) | **انتخاب اول در ۹۰٪ پروژه‌های RAG** |
| **`MarkdownHeaderTextSplitter`** | سطوح تیترهای مارک‌داون (`#`, `##`) | ⚡ بسیار سریع و رایگان | 🟢 خوب (تزریق تیترها به متادیتا) | اسناد ساخت‌یافته، ویکی‌ها و مستندات |
| **`SemanticChunker`** | تغییر موضوع بر پایه Embedding | ⏳ کندتر و نیازمند مدل Embedding | 🟢 عالی (چانک‌ها کاملاً تک‌موضوعی هستند) | مقالات طولانی بدون تیتر که چندین موضوع دارند |

### 🎯 جمع‌بندی نهایی فصل ۱۳ (پردازش داده‌ها در LangChain):

پرونده فصل ۱۳ را با موفقیت بستیم. اکنون بر تمام مراحل پیش‌پردازش داده‌ها مسلط هستیم:
1. **آناتومی کلاس `Document`:** محتوای متنی (`page_content`) + شناسنامه (`metadata`).
2. **بارگذاری اسناد (Loaders):** فایل‌های متنی با `TextLoader`، اسناد PDF با `PyPDFLoader` و استخراج تمیز صفحات وب با `WebBaseLoader` و `SoupStrainer`.
3. **مهندسی داده‌های کلان:** تکنیک `Lazy Loading` برای پردازش جریانی بدون سرریز RAM.
4. **خرد کردن ساخت‌یافته:** تسلط بر `RecursiveCharacterTextSplitter` و بهینه‌سازی برای زبان فارسی و نیم‌فاصله‌ها.
5. **حفظ بافتار:** استفاده از `chunk_overlap` برای رفع تراژدی ضمیرهای سرگردان و `MarkdownHeaderTextSplitter` برای متادیتای بافتی.
6. **تقسیم‌بندی معنایی:** استفاده از `SemanticChunker` برای برش بر اساس پرش‌های موضوعی.

🏁 **آماده برای گام بزرگ بعدی:**
در فصل آینده وارد **فصل ۱۴: دانش خارجی و سیستم‌های RAG (External Knowledge and RAG)** می‌شویم تا قطعات این پازل را به مدل‌های زبانی متصل کرده و یک سیستم هوشمند کامل بسازیم.